In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


In [ ]:
import os
import sys
import random
import copy
from dataclasses import dataclass, field
import warnings
from itertools import product
from typing import Any, Sequence
from pathlib import Path
import pandas as pd
import numpy as np
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import json
import time
from torch import nn



# =====================================================================
# Reproducibility
# =====================================================================
def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# These settings improve reproducibility on CUDA.
if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def resolve_device(
    device: str | torch.device | None = None,
) -> torch.device:
    """
    Resolve the requested PyTorch device.

    Automatic priority:
        CUDA -> Apple MPS -> CPU
    """
    if device is not None:
        resolved_device = torch.device(device)

        if resolved_device.type == "cuda" and not torch.cuda.is_available():
            raise ValueError("CUDA was requested but is not available.")

        if resolved_device.type == "mps":
            mps_available = (
                hasattr(torch.backends, "mps")
                and torch.backends.mps.is_available()
            )

            if not mps_available:
                raise ValueError("MPS was requested but is not available.")

        return resolved_device

    if torch.cuda.is_available():
        return torch.device("cuda")

    if (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
    ):
        return torch.device("mps")

    return torch.device("cpu")

# =====================================================================


# =====================================================================
# Data helpers
# =====================================================================
def reconstruct_chronological_series(
    x: np.ndarray,
    y: np.ndarray,
    slide: int,
) -> np.ndarray:
    """
    Reconstruct the chronological time series represented by overlapping
    supervised-learning rows.

    Example with slide=1:

        row 0 X: t0, ..., t249
        row 1 X: t1, ..., t250

    Therefore, row 1 contributes only its final new observation.

    The final Y row is appended because it contains the known observations
    after the final X window.
    """
    if x.ndim != 2 or y.ndim != 2:
        raise ValueError("x and y must both be two-dimensional arrays.")

    if len(x) == 0:
        raise ValueError("Cannot reconstruct a series from an empty array.")

    if slide < 1:
        raise ValueError("slide must be positive.")

    if slide > x.shape[1]:
        raise ValueError(
            "slide cannot exceed the number of input time steps."
        )

    reconstructed = list(x[0])

    for row_index in range(1, len(x)):
        reconstructed.extend(x[row_index, -slide:])

    reconstructed.extend(y[-1])

    return np.asarray(reconstructed, dtype=np.float64)



def fit_univariate_scaler(
    x: np.ndarray,
    y: np.ndarray,
    slide: int,
) -> StandardScaler:
    """
    Fit one StandardScaler to the reconstructed chronological training
    series only.
    """
    chronological_series = reconstruct_chronological_series(
        x=x,
        y=y,
        slide=slide,
    )

    scaler = StandardScaler()
    scaler.fit(chronological_series.reshape(-1, 1))

    return scaler


def transform_matrix(
    scaler: StandardScaler,
    values: np.ndarray,
) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)

    return scaler.transform(
        values.reshape(-1, 1)
    ).reshape(values.shape)


def inverse_transform_matrix(
    scaler: StandardScaler,
    values: np.ndarray,
) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)

    return scaler.inverse_transform(
        values.reshape(-1, 1)
    ).reshape(values.shape)


def create_data_loader(
    x: np.ndarray,
    y: np.ndarray | None,
    batch_size: int,
    shuffle: bool,
    random_seed: int,
) -> DataLoader:
    """
    Convert NumPy arrays to a PyTorch DataLoader.

    LSTM input shape:
        [number of samples, look_back, number of features]

    Because the series is univariate:
        number of features = 1
    """
    x_tensor = torch.from_numpy(
        x.astype(np.float32, copy=False)
    ).unsqueeze(-1)

    if y is None:
        dataset = TensorDataset(x_tensor)
    else:
        y_tensor = torch.from_numpy(
            y.astype(np.float32, copy=False)
        )
        dataset = TensorDataset(x_tensor, y_tensor)

    generator = torch.Generator()
    generator.manual_seed(random_seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        drop_last=False,
        generator=generator if shuffle else None,
    )


# =====================================================================
# Direct multi-output LSTM
# =====================================================================
class MultiStepLSTM(nn.Module):
    """
    Sequence-to-vector LSTM.

    Input:
        [batch_size, look_back, 1]

    Output:
        [batch_size, horizon]

    A single input sequence therefore produces the complete 20-day
    forecast vector directly.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int,
        horizon: int,
        dropout: float,
    ) -> None:
        super().__init__()

        # PyTorch applies recurrent dropout only between stacked LSTM
        # layers. It has no effect when num_layers=1.
        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )

        self.output_layer = nn.Linear(
            in_features=hidden_dim,
            out_features=horizon,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Run the input sequence through the LSTM and map the final hidden
        state to all forecast horizons.
        """
        if x.ndim == 2:
            x = x.unsqueeze(-1)

        _, (hidden_state, _) = self.lstm(x)

        # Final hidden state from the uppermost LSTM layer
        final_hidden_state = hidden_state[-1]

        return self.output_layer(final_hidden_state)


# =====================================================================
# Training helpers
# =====================================================================
def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_function: nn.Module,
    device: torch.device,
    gradient_clip: float | None,
) -> float:
    """
    Train the model for one epoch and return average MSE.
    """
    model.train()

    loss_sum = 0.0
    observation_count = 0

    for batch_x, batch_y in data_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad(set_to_none=True)

        predictions = model(batch_x)
        loss = loss_function(predictions, batch_y)

        loss.backward()

        if gradient_clip is not None:
            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=gradient_clip,
            )

        optimizer.step()

        current_batch_size = batch_x.size(0)

        loss_sum += loss.item() * current_batch_size
        observation_count += current_batch_size

    if observation_count == 0:
        raise RuntimeError("No training observations were processed.")

    average_loss = loss_sum / observation_count

    if not np.isfinite(average_loss):
        raise RuntimeError(
            "The training loss became NaN or infinite."
        )

    return average_loss


def evaluate_validation_loss(
    model: nn.Module,
    data_loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
) -> float:
    """
    Evaluate validation MSE without updating model parameters.
    """
    model.eval()

    loss_sum = 0.0
    observation_count = 0

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            predictions = model(batch_x)
            loss = loss_function(predictions, batch_y)

            current_batch_size = batch_x.size(0)

            loss_sum += loss.item() * current_batch_size
            observation_count += current_batch_size

    if observation_count == 0:
        raise RuntimeError("No validation observations were processed.")

    average_loss = loss_sum / observation_count

    if not np.isfinite(average_loss):
        raise RuntimeError(
            "The validation loss became NaN or infinite."
        )

    return average_loss


def fit_candidate_model(
    train_x: np.ndarray,
    train_y: np.ndarray,
    validation_x: np.ndarray,
    validation_y: np.ndarray,
    horizon: int,
    hidden_dim: int,
    num_layers: int,
    dropout: float,
    learning_rate: float,
    batch_size: int,
    max_epochs: int,
    patience: int,
    min_delta: float,
    weight_decay: float,
    gradient_clip: float | None,
    device: torch.device,
    random_seed: int,
) -> dict:
    """
    Fit one candidate LSTM architecture using chronological validation
    and early stopping.

    The returned best epoch is subsequently used to retrain the selected
    architecture on the complete training dataset.
    """
    set_random_seed(random_seed)

    train_loader = create_data_loader(
        x=train_x,
        y=train_y,
        batch_size=batch_size,
        shuffle=True,
        random_seed=random_seed,
    )

    validation_loader = create_data_loader(
        x=validation_x,
        y=validation_y,
        batch_size=batch_size,
        shuffle=False,
        random_seed=random_seed,
    )

    model = MultiStepLSTM(
        input_dim=1,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        horizon=horizon,
        dropout=dropout,
    ).to(device)

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    best_validation_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epochs_trained = 0

    candidate_start_time = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        train_one_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            loss_function=loss_function,
            device=device,
            gradient_clip=gradient_clip,
        )

        validation_loss = evaluate_validation_loss(
            model=model,
            data_loader=validation_loader,
            loss_function=loss_function,
            device=device,
        )

        epochs_trained = epoch

        if validation_loss < best_validation_loss - min_delta:
            best_validation_loss = validation_loss
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    candidate_training_time = (
        time.perf_counter() - candidate_start_time
    )

    if best_epoch == 0:
        raise RuntimeError(
            f"No valid model was obtained for hidden_dim={hidden_dim}."
        )

    return {
        "hidden_dim": int(hidden_dim),
        "best_epoch": int(best_epoch),
        "epochs_trained": int(epochs_trained),
        "best_validation_mse": float(
            best_validation_loss
        ),
        "training_time_seconds": float(
            candidate_training_time
        ),
    }


def train_final_model(
    train_x: np.ndarray,
    train_y: np.ndarray,
    horizon: int,
    hidden_dim: int,
    num_layers: int,
    dropout: float,
    learning_rate: float,
    batch_size: int,
    epochs: int,
    weight_decay: float,
    gradient_clip: float | None,
    device: torch.device,
    random_seed: int,
) -> tuple[nn.Module, float]:
    """
    Retrain the selected architecture from scratch using all available
    training rows.

    The number of epochs is selected by the validation procedure.
    """
    if epochs < 1:
        raise ValueError("epochs must be positive.")

    set_random_seed(random_seed)

    train_loader = create_data_loader(
        x=train_x,
        y=train_y,
        batch_size=batch_size,
        shuffle=True,
        random_seed=random_seed,
    )

    model = MultiStepLSTM(
        input_dim=1,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        horizon=horizon,
        dropout=dropout,
    ).to(device)

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    final_training_loss = float("nan")

    for _ in range(epochs):
        final_training_loss = train_one_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            loss_function=loss_function,
            device=device,
            gradient_clip=gradient_clip,
        )

    return model, final_training_loss


# =====================================================================
# Prediction helper
# =====================================================================
def predict_multistep(
    model: nn.Module,
    x: np.ndarray,
    batch_size: int,
    device: torch.device,
    random_seed: int,
) -> np.ndarray:
    """
    Generate one direct multi-horizon forecast for each test row.
    """
    data_loader = create_data_loader(
        x=x,
        y=None,
        batch_size=batch_size,
        shuffle=False,
        random_seed=random_seed,
    )

    model.eval()

    prediction_batches = []

    with torch.no_grad():
        for (batch_x,) in data_loader:
            batch_x = batch_x.to(device)

            batch_predictions = model(batch_x)

            prediction_batches.append(
                batch_predictions.cpu().numpy()
            )

    if not prediction_batches:
        raise RuntimeError("No test predictions were generated.")

    predictions = np.concatenate(
        prediction_batches,
        axis=0,
    )

    if not np.isfinite(predictions).all():
        raise RuntimeError(
            "The generated predictions contain NaN or infinite values."
        )

    return predictions


# =====================================================================
# Multistep LSTM forecasting: training and testing
# =====================================================================
def trainAndtest(
    trainf: str,
    testf: str,
    look_back: int,
    horizon: int,
    slide: int,
    outf: str,
    txt_outf: str,
    hidden_dims: Sequence[int] = (4, 8, 16, 32),
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    batch_size: int = 64,
    max_epochs: int = 200,
    patience: int = 20,
    min_delta: float = 1e-5,
    validation_fraction: float = 0.15,
    purge_validation: bool = True,
    weight_decay: float = 0.0,
    gradient_clip: float | None = 1.0,
    random_seed: int = 42,
    device: str | torch.device | None = None,
    verbose: bool = True,
) -> None:
    """
    Train, select, and test a direct multi-output LSTM.

    Parameters
    ----------
    trainf
        Training CSV containing X_0,...,X_{look_back-1} and
        Y_0,...,Y_{horizon-1}.

    testf
        Testing CSV with the same supervised column structure.

    look_back
        Number of observations in each input sequence.

    horizon
        Number of future observations predicted simultaneously.

    slide
        Distance, in observations, between consecutive supervised rows.

    outf
        CSV file to which test inputs and predictions are written.

    txt_outf
        JSON file to which hyperparameters, timings, and metrics are
        written.

    hidden_dims
        Candidate LSTM hidden-state dimensions. Selection is performed
        using chronological validation only.

    validation_fraction
        Fraction of the training rows assigned to the final chronological
        validation block.

    purge_validation
        When True, remove rows between model-fitting and validation
        blocks so that their output horizons do not overlap.

    Notes
    -----
    X and Y are standardized with one univariate StandardScaler fitted only
    on chronological training data. During model selection it is fitted only
    on the model-fitting subset; after selection a new scaler is fitted on
    the complete training partition. Test Y values never contribute to
    scaling, fitting, or prediction. Forecasts are inverse-transformed before
    metrics are calculated or CSVs are saved.
    """

    # ---------------------------------------------------------------
    # Validate configuration
    # ---------------------------------------------------------------
    if not os.path.exists(trainf):
        raise FileNotFoundError(f"{trainf} does not exist.")

    if not os.path.exists(testf):
        raise FileNotFoundError(f"{testf} does not exist.")

    if look_back < 1:
        raise ValueError("look_back must be positive.")

    if horizon < 1:
        raise ValueError("horizon must be positive.")

    if slide < 1:
        raise ValueError("slide must be positive.")

    if slide > look_back:
        raise ValueError("slide cannot exceed look_back.")

    if not 0.0 < validation_fraction < 1.0:
        raise ValueError(
            "validation_fraction must be strictly between 0 and 1."
        )

    hidden_dims = tuple(int(value) for value in hidden_dims)

    if len(hidden_dims) == 0:
        raise ValueError("At least one hidden dimension is required.")

    if any(value < 1 for value in hidden_dims):
        raise ValueError(
            "Every value in hidden_dims must be positive."
        )

    if num_layers < 1:
        raise ValueError("num_layers must be positive.")

    if not 0.0 <= dropout < 1.0:
        raise ValueError("dropout must be in [0, 1).")

    if learning_rate <= 0.0:
        raise ValueError("learning_rate must be positive.")

    if batch_size < 1:
        raise ValueError("batch_size must be positive.")

    if max_epochs < 1:
        raise ValueError("max_epochs must be positive.")

    if patience < 1:
        raise ValueError("patience must be positive.")

    if min_delta < 0.0:
        raise ValueError("min_delta cannot be negative.")

    if weight_decay < 0.0:
        raise ValueError("weight_decay cannot be negative.")

    if gradient_clip is not None and gradient_clip <= 0.0:
        raise ValueError(
            "gradient_clip must be positive or None."
        )

    # ---------------------------------------------------------------
    # Load supervised data
    # ---------------------------------------------------------------
    train_df = pd.read_csv(trainf)
    test_df = pd.read_csv(testf)

    if verbose:
        print(
            "# Raw shapes of train and test:",
            train_df.shape,
            test_df.shape,
        )

    x_cols = [f"X_{i}" for i in range(look_back)]
    y_cols = [f"Y_{i}" for i in range(horizon)]

    required_columns = x_cols + y_cols

    for dataset_name, dataframe in (
        ("training", train_df),
        ("testing", test_df),
    ):
        missing_columns = [
            column
            for column in required_columns
            if column not in dataframe.columns
        ]

        if missing_columns:
            raise ValueError(
                f"The {dataset_name} data are missing required "
                f"columns: {missing_columns}"
            )

    trainX_df = train_df[x_cols]
    trainY_df = train_df[y_cols]
    testX_df = test_df[x_cols]
    testY_df = test_df[y_cols]

    trainX = trainX_df.to_numpy(dtype=np.float64)
    trainY = trainY_df.to_numpy(dtype=np.float64)
    testX = testX_df.to_numpy(dtype=np.float64)
    testY = testY_df.to_numpy(dtype=np.float64)

    if verbose:
        print(
            "# Supervised shapes of train X, train Y, test X, test Y:",
            trainX.shape,
            trainY.shape,
            testX.shape,
            testY.shape,
        )

    for array_name, values in (
        ("trainX", trainX),
        ("trainY", trainY),
        ("testX", testX),
        ("testY", testY),
    ):
        if values.size == 0:
            raise ValueError(f"{array_name} is empty.")

        if not np.isfinite(values).all():
            raise ValueError(
                f"{array_name} contains NaN or infinite values."
            )

    # ---------------------------------------------------------------
    # Chronological training-validation split
    # ---------------------------------------------------------------
    number_of_train_rows = len(trainX)

    number_of_validation_rows = max(
        1,
        int(
            math.ceil(
                number_of_train_rows * validation_fraction
            )
        ),
    )

    validation_start = (
        number_of_train_rows - number_of_validation_rows
    )

    # Consecutive supervised rows can have overlapping output labels.
    #
    # For horizon=20 and slide=1:
    #   purge_rows = ceil(20 / 1) - 1 = 19
    #
    # This ensures that training and validation Y periods do not overlap.
    if purge_validation:
        purge_rows = max(
            0,
            math.ceil(horizon / slide) - 1,
        )
    else:
        purge_rows = 0

    model_fit_end = validation_start - purge_rows

    if model_fit_end < 2:
        raise ValueError(
            "Insufficient training rows after constructing the "
            "validation split. "
            f"Total rows={number_of_train_rows}, "
            f"validation rows={number_of_validation_rows}, "
            f"purged rows={purge_rows}."
        )

    model_fit_x = trainX[:model_fit_end]
    model_fit_y = trainY[:model_fit_end]

    validation_x = trainX[validation_start:]
    validation_y = trainY[validation_start:]

    if verbose:
        print("# Model-selection training rows:", len(model_fit_x))
        print("# Purged rows:", purge_rows)
        print("# Validation rows:", len(validation_x))


    # ---------------------------------------------------------------
    # StandardScaler for chronological MODEL SELECTION
    # ---------------------------------------------------------------
    # Fit only on the model-fitting block. Validation/test data do not
    # contribute to the scaler statistics.
    selection_scaler = fit_univariate_scaler(
        x=model_fit_x,
        y=model_fit_y,
        slide=slide,
    )

    model_fit_x_scaled = transform_matrix(
        selection_scaler,
        model_fit_x,
    )
    model_fit_y_scaled = transform_matrix(
        selection_scaler,
        model_fit_y,
    )
    validation_x_scaled = transform_matrix(
        selection_scaler,
        validation_x,
    )
    validation_y_scaled = transform_matrix(
        selection_scaler,
        validation_y,
    )

    if verbose:
        print(
            "# Selection scaler mean:",
            float(selection_scaler.mean_[0]),
        )
        print(
            "# Selection scaler scale:",
            float(selection_scaler.scale_[0]),
        )


    

    # ---------------------------------------------------------------
    # Select the LSTM hidden dimension
    # ---------------------------------------------------------------
    resolved_device = resolve_device(device)

    if verbose:
        print("# PyTorch device:", resolved_device)
        print("# Candidate hidden dimensions:", hidden_dims)

    candidate_results = []

    selection_start_time = time.perf_counter()

    for hidden_dim in hidden_dims:
        if verbose:
            print(
                f"Training candidate LSTM with "
                f"hidden_dim={hidden_dim}..."
            )

        candidate_result = fit_candidate_model(
            train_x=model_fit_x_scaled,
            train_y=model_fit_y_scaled,
            validation_x=validation_x_scaled,
            validation_y=validation_y_scaled,
            horizon=horizon,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            learning_rate=learning_rate,
            batch_size=batch_size,
            max_epochs=max_epochs,
            patience=patience,
            min_delta=min_delta,
            weight_decay=weight_decay,
            gradient_clip=gradient_clip,
            device=resolved_device,
            random_seed=random_seed,
        )

        validation_mse = candidate_result[
            "best_validation_mse"
        ]

        validation_rmse = math.sqrt(validation_mse)
        
        candidate_result[
            "best_validation_rmse"
        ] = float(validation_rmse)
        
        candidate_results.append(candidate_result)

        if verbose:
            print(
                "  Best epoch:",
                candidate_result["best_epoch"],
            )
            print(
                "  Validation RMSE:",
                validation_rmse,
            )

    selection_time_seconds = (
        time.perf_counter() - selection_start_time
    )

    best_candidate = min(
        candidate_results,
        key=lambda result: result[
            "best_validation_mse"
        ],
    )

    selected_hidden_dim = int(
        best_candidate["hidden_dim"]
    )

    selected_epochs = int(
        best_candidate["best_epoch"]
    )

    if verbose:
        print("# Selected hidden dimension:", selected_hidden_dim)
        print("# Selected training epochs:", selected_epochs)

    # ---------------------------------------------------------------
    # FINAL StandardScaler
    # ---------------------------------------------------------------
    # Refit a fresh scaler on the COMPLETE 80% training partition.
    # The 20% held-out test partition remains completely untouched.
    final_scaler = fit_univariate_scaler(
        x=trainX,
        y=trainY,
        slide=slide,
    )

    trainX_scaled = transform_matrix(
        final_scaler,
        trainX,
    )
    trainY_scaled = transform_matrix(
        final_scaler,
        trainY,
    )
    testX_scaled = transform_matrix(
        final_scaler,
        testX,
    )

    if verbose:
        print(
            "# Final scaler mean:",
            float(final_scaler.mean_[0]),
        )
        print(
            "# Final scaler scale:",
            float(final_scaler.scale_[0]),
        )

    # ---------------------------------------------------------------
    # Train the final LSTM on the complete STANDARDIZED training dataset
    # ---------------------------------------------------------------
    final_fit_start_time = time.perf_counter()

    final_model, final_training_mse = train_final_model(
        train_x=trainX_scaled,
        train_y=trainY_scaled,
        horizon=horizon,
        hidden_dim=selected_hidden_dim,
        num_layers=num_layers,
        dropout=dropout,
        learning_rate=learning_rate,
        batch_size=batch_size,
        epochs=selected_epochs,
        weight_decay=weight_decay,
        gradient_clip=gradient_clip,
        device=resolved_device,
        random_seed=random_seed,
    )

    final_fit_time_seconds = (
        time.perf_counter() - final_fit_start_time
    )

    training_time_seconds = (
        selection_time_seconds + final_fit_time_seconds
    )
    
    model_parameter_count = sum(
        parameter.numel()
        for parameter in final_model.parameters()
        if parameter.requires_grad
    )
    
    if verbose:
        print("# Trainable model parameters:", model_parameter_count)
        print("# Model-selection time:", selection_time_seconds)
        print("# Final fitting time:", final_fit_time_seconds)
        print("# Total training time:", training_time_seconds)

    # ---------------------------------------------------------------
    # Forecast every test row
    # ---------------------------------------------------------------
    # Each test row contains the complete observed look-back sequence.
    # The training-fitted StandardScaler transforms test X. The LSTM
    # predicts standardized horizons, which are inverse-transformed before
    # metrics or files are produced.
    #
    # testY is not used in scaling, fitting, or prediction.
    # ---------------------------------------------------------------
    test_start_time = time.perf_counter()

    pred_arr_scaled = predict_multistep(
        model=final_model,
        x=testX_scaled,
        batch_size=batch_size,
        device=resolved_device,
        random_seed=random_seed,
    )

    # Return forecasts to the ORIGINAL scale before all metrics and saving.
    pred_arr = inverse_transform_matrix(
        final_scaler,
        pred_arr_scaled,
    )
    
    test_end_time = time.perf_counter()

    testing_time_seconds = (
        test_end_time - test_start_time
    )

    if pred_arr.shape != testY.shape:
        raise RuntimeError(
            "Prediction shape does not match the target shape: "
            f"predictions={pred_arr.shape}, targets={testY.shape}."
        )

    # ---------------------------------------------------------------
    # Compute metrics on the original data scale
    # ---------------------------------------------------------------
    overall_mae = mean_absolute_error(
        testY.ravel(),
        pred_arr.ravel(),
    )

    overall_rmse = np.sqrt(
        mean_squared_error(
            testY.ravel(),
            pred_arr.ravel(),
        )
    )

    horizon_mae = []
    horizon_rmse = []

    for horizon_index in range(horizon):
        current_mae = mean_absolute_error(
            testY[:, horizon_index],
            pred_arr[:, horizon_index],
        )

        current_rmse = np.sqrt(
            mean_squared_error(
                testY[:, horizon_index],
                pred_arr[:, horizon_index],
            )
        )

        horizon_mae.append(float(current_mae))
        horizon_rmse.append(float(current_rmse))

    if verbose:
        print("Overall MAE :", overall_mae)
        print("Overall RMSE:", overall_rmse)
        print("Testing time seconds:", testing_time_seconds)

    # ---------------------------------------------------------------
    # Save predictions
    # ---------------------------------------------------------------
    output_directory = os.path.dirname(
        os.path.abspath(outf)
    )

    results_directory = os.path.dirname(
        os.path.abspath(txt_outf)
    )

    os.makedirs(output_directory, exist_ok=True)
    os.makedirs(results_directory, exist_ok=True)

    # Retain test X columns and replace test Y with forecasts.
    out_df = testX_df.copy()

    for horizon_index, column in enumerate(y_cols):
        out_df[column] = pred_arr[:, horizon_index]

    out_df.to_csv(outf, index=False)

    # ---------------------------------------------------------------
    # Save full model-selection and evaluation results
    # ---------------------------------------------------------------
    results_to_save = {
        "model": "DirectMultiStepLSTM",
        "forecasting_strategy": "direct_multi_output",
        "look_back": int(look_back),
        "horizon": int(horizon),
        "slide": int(slide),
        "device": str(resolved_device),
        "random_seed": int(random_seed),

        "scaling": {
            "type": "StandardScaler",
            "fit_scope": (
                "single univariate scaler fitted only on "
                "chronological training data"
            ),
            "selection_scaler_fit_on_model_fit_only": True,
            "final_scaler_mean": float(
                final_scaler.mean_[0]
            ),
            "final_scaler_scale": float(
                final_scaler.scale_[0]
            ),
            "inverse_transform_before_metrics_and_saving": True,
        },

        "architecture": {
            "input_dim": 1,
            "selected_hidden_dim": selected_hidden_dim,
            "num_layers": int(num_layers),
            "dropout": float(dropout),
            "trainable_parameters": int(
                model_parameter_count
            ),
        },

        "optimization": {
            "optimizer": "Adam",
            "loss_function": "MSELoss",
            "learning_rate": float(learning_rate),
            "weight_decay": float(weight_decay),
            "batch_size": int(batch_size),
            "gradient_clip": (
                None
                if gradient_clip is None
                else float(gradient_clip)
            ),
            "maximum_epochs": int(max_epochs),
            "selected_epochs": selected_epochs,
            "patience": int(patience),
            "min_delta": float(min_delta),
        },

        "validation": {
            "method": "chronological_holdout",
            "validation_fraction": float(
                validation_fraction
            ),
            "purge_validation": bool(
                purge_validation
            ),
            "model_fit_rows": int(model_fit_end),
            "purged_rows": int(purge_rows),
            "validation_rows": int(
                number_of_validation_rows
            ),
            "candidate_results": candidate_results,
        },

        "final_training_mse_standardized_scale": float(
            final_training_mse
        ),

        "selection_time_seconds": float(
            selection_time_seconds
        ),

        "final_fit_time_seconds": float(
            final_fit_time_seconds
        ),

        "training_time_seconds": float(
            training_time_seconds
        ),

        "testing_time_seconds": float(
            testing_time_seconds
        ),

        "overall_mae": float(overall_mae),
        "overall_rmse": float(overall_rmse),

        "horizon_mae": horizon_mae,
        "horizon_rmse": horizon_rmse,
    }

    with open(txt_outf, "w", encoding="utf-8") as file:
        json.dump(
            results_to_save,
            file,
            indent=4,
        )

    print(f"Saved forecasts to: {outf}")
    print(f"Saved full results to: {txt_outf}")

    return



# =====================================================================
# Asset-class runner
# =====================================================================
if __name__ == "__main__":

    DATA_DIR = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")

    RESULTS_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/lstm_standardized_train_test")

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    ASSET_FOLDERS = {
        "G7": "FX",
        "EM": "EM FX",
        "MSCI": "MSCI",
        "Rates": "Rates",
        "REIT": "REIT",
        "Equities": "Equities",
        "ETF": "ETF",
        "Crypto": "Crypto",
    }

    model = "lstm"
    look_back = 250
    horizon = 20
    slide = 1

    for folder_name, asset_class in ASSET_FOLDERS.items():

        input_dir = DATA_DIR / folder_name
        output_dir = RESULTS_DIR / folder_name

        output_dir.mkdir(parents=True, exist_ok=True)

        if not input_dir.exists():
            print(f"WARNING: missing input folder: {input_dir}")
            continue

        pattern = (
            f"train_*_no_preprocess"
            f"_lookback-{look_back}"
            f"_horizon-{horizon}"
            f"_slide-{slide}.csv"
        )

        train_paths = sorted(input_dir.glob(pattern))

        print("\n" + "=" * 80)
        print(f"ASSET CLASS: {asset_class} ({folder_name})")
        print(f"FOUND {len(train_paths)} TRAIN FILES")
        print("=" * 80)

        for train_csv in train_paths:

            test_name = "test_" + train_csv.name[len("train_"):]
            test_csv = input_dir / test_name

            if not test_csv.exists():
                print(
                    f"WARNING: missing matching test file "
                    f"for {train_csv.name}"
                )
                continue

            test_stem = test_csv.stem

            forecasted_csv = (
                output_dir
                / f"{test_stem}_{model}.csv"
            )

            results_json = (
                output_dir
                / f"{test_stem}_{model}.json"
            )

            print("\n" + "-" * 80)
            print(f"TRAIN: {train_csv.name}")
            print(f"TEST:  {test_csv.name}")
            print(f"OUT:   {forecasted_csv.name}")
            print("-" * 80)

            trainAndtest(
                trainf=str(train_csv),
                testf=str(test_csv),
                look_back=look_back,
                horizon=horizon,
                slide=slide,
                outf=str(forecasted_csv),
                txt_outf=str(results_json),
                hidden_dims=(4, 8, 16, 32),
                num_layers=1,
                dropout=0.0,
                learning_rate=1e-3,
                batch_size=64,
                max_epochs=500,
                patience=20,
                min_delta=1e-5,
                gradient_clip=1.0,
                validation_fraction=0.15,
                purge_validation=True,
                random_seed=42,
                device=None,
                verbose=False,
            )

    print("\nALL LSTM DATASETS COMPLETE.")
